# Adia Lab Causal Discovery — 1st Place Solution

Full implementation of the top solution described at [thetourney.github.io/adia-report](https://thetourney.github.io/adia-report/).

### Architecture
- **Preprocessing**: Edge tensors with 3 channels (sorted u, sorted v, kernel regression coefficient)
- **Stem layer**: 3→64 channels linear
- **5× Residual Conv1D blocks**: GroupNorm + GELU + residual
- **Average pooling**: → 64-dim edge embeddings
- **Merge operator**: edge emb + edge-type embedding → LayerNorm + GELU
- **2× Self-attention layers**
- **Edge head** (training only): binary adjacency matrix
- **Node head**: gather 4 edges (u→X, u→Y, X→u, Y→u) → 8-class classification
- **Training**: AdamW + Cosine Annealing, inverse-frequency class weights, dual CE loss

### Setup

In [1]:
# %pip install pytorch_lightning
# %pip install crunch-cli --upgrade --quiet

In [2]:
# %pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch --env staging setup-notebook causality-discovery 0LieCWz8NrSdOTsZFGE39SaV --no-data

environment: forcing staging urls, ignoring $API_BASE_URL and $WEB_BASE_URL
crunch-cli, version 11.0.0



---
Your token seems to have expired or is invalid.

Please follow this link to copy and paste your new setup command:
https://hub.crunchdao.io/competitions/causality-discovery/submit

If you think that is an error, please contact an administrator.


In [3]:
%env API_BASE_URL=https://api.hub.crunchdao.io/
%env WEB_BASE_URL=https://hub.crunchdao.io/

import crunch.store
crunch.store.load_from_env()

env: API_BASE_URL=https://api.hub.crunchdao.io/
env: WEB_BASE_URL=https://hub.crunchdao.io/


### Imports & Setup

In [4]:
import typing
import os
from tqdm.auto import tqdm

import pandas as pd
import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import pytorch_lightning as pl

import networkx as nx
from sklearn.model_selection import train_test_split
from multiprocessing import Pool, cpu_count

### Data Preprocessing

In [5]:
# @crunch/keep:on

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_OBS = 1000
N_KERNEL = 150   # subsample for kernel regression — major speedup
MAX_VARS = 10
N_CLASSES = 8
N_EDGE_TYPES = 7
D_MODEL = 64
CLASS_NAMES = [
    "Confounder", "Collider", "Mediator",
    "Cause of X", "Cause of Y",
    "Consequence of X", "Consequence of Y",
    "Independent",
]


def _edge_type(u_name, v_name):
    uX, uY = u_name == "X", u_name == "Y"
    vX, vY = v_name == "X", v_name == "Y"
    if uX and not vY:  return 0
    if uX and vY:      return 1
    if uY and not vX:  return 2
    if uY and vX:      return 3
    if not uX and not uY and vX: return 4
    if not uX and not uY and vY: return 5
    return 6


def _build_single(args):
    df, y_df = args                          # DataFrames directly, no dict lookup
    edge_data, edge_types = build_edge_tensor(df)
    result = {
        "edge_data":  edge_data,
        "edge_types": edge_types,
        "cols": list(df.columns),
        "p": len(df.columns),
    }
    if y_df is not None:
        result["adj"] = y_df.values.astype(np.float32)
        result["adj_cols"] = list(y_df.columns)
    return result

In [6]:
def compute_kernel_coeff_fast(
    u: np.ndarray,  # (N,)
    v: np.ndarray,  # (N,)
    n_sub: int = N_KERNEL,
    bw: float = 0.5,
) -> np.ndarray:
    """
    Fast pairwise kernel regression coefficient of v on u.
    Subsample to n_sub points for the weight matrix, then interpolate back.
    Returns (N,) coefficients.
    """
    N = len(u)
    idx = np.random.choice(N, min(n_sub, N), replace=False)
    u_s, v_s = u[idx], v[idx]  # (n_sub,)

    # (n_sub, n_sub) weight matrix — much smaller than (N, N)
    diff = u_s[:, None] - u_s[None, :]    # (n_sub, n_sub)
    w = np.exp(-diff ** 2 / (2 * bw ** 2))  # (n_sub, n_sub)
    w_sum = w.sum(axis=1)                    # (n_sub,)

    wu = (w * u_s[None, :]).sum(axis=1)      # (n_sub,)
    wv = (w * v_s[None, :]).sum(axis=1)      # (n_sub,)
    wuu = (w * (u_s[None, :] ** 2)).sum(axis=1)
    wuv = (w * u_s[None, :] * v_s[None, :]).sum(axis=1)

    denom = wuu - wu ** 2 / w_sum
    numer = wuv - wu * wv / w_sum
    coeff_sub = np.where(np.abs(denom) > 1e-8, numer / denom, 0.0)

    # Interpolate back to full N via nearest-neighbor in u-space
    # For each observation, find the closest subsampled point
    dists = np.abs(u[:, None] - u_s[None, :])  # (N, n_sub)
    nearest = np.argmin(dists, axis=1)           # (N,)
    return coeff_sub[nearest].astype(np.float32)


def build_edge_tensor(
    df: pd.DataFrame,
) -> tuple:
    """
    Given a DataFrame of shape (1000, p), build:
      - edge_data: shape (p*(p-1), 3, 1000)
      - edge_types: shape (p*(p-1),) — integer 0-6
    """
    cols = list(df.columns)
    p = len(cols)
    data = df.values.astype(np.float32)  # (1000, p)
    N = data.shape[0]

    edges = []
    edge_types = []

    for i, u_name in enumerate(cols):
        sort_idx = np.argsort(data[:, i])
        u_sorted = data[sort_idx, i]  # (N,)

        for j, v_name in enumerate(cols):
            if i == j:
                continue

            v_sorted_by_u = data[sort_idx, j]  # (N,)

            # Fast subsampled kernel regression
            coeff_all = compute_kernel_coeff_fast(data[:, i], data[:, j])  # (N,)
            coeff_sorted = coeff_all[sort_idx]  # (N,)

            edge_tensor = np.stack(
                [u_sorted, v_sorted_by_u, coeff_sorted], axis=0
            )  # (3, N)
            edges.append(edge_tensor)
            edge_types.append(_edge_type(u_name, v_name))

    edge_data = np.stack(edges, axis=0).astype(np.float32)
    edge_types = np.array(edge_types, dtype=np.int64)
    return edge_data, edge_types


def build_node_index_map(cols: list) -> dict:
    """Return {name: index} for the column list."""
    return {name: i for i, name in enumerate(cols)}


def get_edge_index(u_idx: int, v_idx: int, p: int) -> int:
    """Convert (u, v) node indices to edge list index in the p*(p-1) list."""
    count = 0
    for i in range(p):
        for j in range(p):
            if i == j:
                continue
            if i == u_idx and j == v_idx:
                return count
            count += 1
    raise ValueError(f"Edge ({u_idx},{v_idx}) not found")

### Dataset

In [7]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class CausalEdgeDataset(Dataset):
    def __init__(
        self,
        X_list: list,
        y_list: list = None,
        n_workers: int = None,
    ):
        n_workers = n_workers or max(1, cpu_count() - 1)
        args = [
            (X_list[i], y_list[i] if y_list else None)
            for i in range(len(X_list))
        ]
        print(f"Building edge dataset ({len(args)} samples, {n_workers} workers)...")
        with Pool(n_workers) as pool:
            raw = list(tqdm(pool.imap(_build_single, args, chunksize=8), total=len(args)))

        _, adjacency_label = create_graph_label()
        self.samples = []
        for item in raw:
            sample = {
                "edge_data":  torch.tensor(item["edge_data"]),   # (E, 3, N)
                "edge_types": torch.tensor(item["edge_types"]),  # (E,)
                "cols": item["cols"],
                "p": item["p"],
            }
            if "adj" in item:
                adj_np = item["adj"]
                adj_cols = item["adj_cols"]
                sample["adj"] = torch.tensor(adj_np)

                # Node labels
                df_adj = pd.DataFrame(adj_np, columns=adj_cols, index=adj_cols)
                node_labels = {}
                for v in adj_cols:
                    if v in ("X", "Y"):
                        continue
                    sub = df_adj.loc[[v, "X", "Y"], [v, "X", "Y"]]
                    key = tuple(sub.values.flatten())
                    label_str = adjacency_label.get(key, "Independent")
                    node_labels[v] = CLASS_NAMES.index(label_str)
                sample["node_labels"] = node_labels

            self.samples.append(sample)

    def __len__(self):  return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_fn(batch):
    max_E = max(item["edge_data"].shape[0] for item in batch)
    B = len(batch)

    # Hardcode — never derive from batch[0] since p varies across samples
    edge_data  = torch.zeros(B, max_E, 3, N_OBS)
    edge_types = torch.zeros(B, max_E, dtype=torch.long)
    edge_mask  = torch.zeros(B, max_E, dtype=torch.bool)

    adj_list, node_labels_list, cols_list = [], [], []

    for b, item in enumerate(batch):
        E = item["edge_data"].shape[0]
        edge_data[b, :E]  = item["edge_data"]
        edge_types[b, :E] = item["edge_types"]
        edge_mask[b, :E]  = True
        cols_list.append(item["cols"])
        if "adj" in item:
            adj_list.append(item["adj"])
            node_labels_list.append(item["node_labels"])

    out = {
        "edge_data": edge_data, "edge_types": edge_types,
        "edge_mask": edge_mask, "cols": cols_list,
    }
    if adj_list:
        out["adj"] = adj_list
        out["node_labels"] = node_labels_list
    return out

### Model Architecture

In [8]:
class ConvBlock(nn.Module):
    """
    Residual 1D convolutional block with GroupNorm + GELU.
    Fig. 5 in the report.
    """

    def __init__(self, d: int, kernel_size: int = 3, n_groups: int = 8):
        super().__init__()
        self.conv = nn.Conv1d(
            d, d, kernel_size, padding=kernel_size // 2, bias=False
        )
        self.norm = nn.GroupNorm(n_groups, d)
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*E, d, N)
        return x + self.act(self.norm(self.conv(x)))


class MergeOperator(nn.Module):
    """
    Concatenate inputs on channel dim, reduce back to d_model.
    Fig. 6 in the report.
    """

    def __init__(self, n_inputs: int, d: int):
        super().__init__()
        self.linear = nn.Linear(n_inputs * d, d)
        self.norm = nn.LayerNorm(d)
        self.act = nn.GELU()

    def forward(self, inputs: typing.List[torch.Tensor]) -> torch.Tensor:
        # Each input: (..., d)
        x = torch.cat(inputs, dim=-1)
        return self.act(self.norm(self.linear(x)))


class StemLayer(nn.Module):
    """Linear layer expanding 3 channels -> d_model."""

    def __init__(self, d: int):
        super().__init__()
        self.linear = nn.Linear(3, d)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*E, 3, N)
        # Transpose to (B*E, N, 3), apply linear, transpose back
        return self.linear(x.permute(0, 2, 1)).permute(0, 2, 1)


class EdgeFeatureExtractor(nn.Module):
    """
    Stem + 5 conv blocks + average pooling -> (B*E, d_model).
    """

    def __init__(self, d: int = D_MODEL, n_blocks: int = 5):
        super().__init__()
        self.stem = StemLayer(d)
        self.blocks = nn.Sequential(*[ConvBlock(d) for _ in range(n_blocks)])
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B*E, 3, N)
        x = self.stem(x)           # (B*E, d, N)
        x = self.blocks(x)         # (B*E, d, N)
        x = self.pool(x).squeeze(-1)  # (B*E, d)
        return x


class SelfAttentionLayer(nn.Module):
    """
    Standard multi-head self-attention over edge embeddings.
    """

    def __init__(self, d: int = D_MODEL, n_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)
        )
        self.norm2 = nn.LayerNorm(d)

    def forward(
        self, x: torch.Tensor, key_padding_mask: torch.Tensor = None
    ) -> torch.Tensor:
        # x: (B, E, d)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class ADIAModel(nn.Module):
    """
    Full 1st-place model as described in https://thetourney.github.io/adia-report/

    Pipeline:
      1. StemLayer:           (B, E, 3, N) -> (B, E, d)   [via EdgeFeatureExtractor]
      2. Edge type embedding:  merge with (B, E, d) type embeddings
      3. 2x SelfAttentionLayer: refine edge embeddings
      4. Edge classification head (training only): (B, E, 2)
      5. Node classification head: gather 4 edges per node -> (B, num_other_nodes, 8)
    """

    def __init__(self, d: int = D_MODEL, n_edge_types: int = N_EDGE_TYPES):
        super().__init__()
        self.d = d
        self.extractor = EdgeFeatureExtractor(d)
        self.edge_type_emb = nn.Embedding(n_edge_types, d)
        self.edge_merge = MergeOperator(n_inputs=2, d=d)
        self.attn_layers = nn.ModuleList([
            SelfAttentionLayer(d) for _ in range(2)
        ])
        # Edge classification head (binary adjacency)
        self.edge_head = nn.Linear(d, 2)
        # Node classification head uses a Merge(4 edges) -> linear(8)
        self.node_merge = MergeOperator(n_inputs=4, d=d)
        self.node_head = nn.Linear(d, N_CLASSES)

    def forward(
        self,
        edge_data: torch.Tensor,    # (B, E, 3, N)
        edge_types: torch.Tensor,   # (B, E)
        edge_mask: torch.Tensor,    # (B, E) True = valid
        cols_list: typing.List[typing.List[str]],
    ) -> typing.Tuple[torch.Tensor, typing.List[torch.Tensor]]:
        """
        Returns:
          edge_logits: (B, E, 2) — for edge classification head
          node_logits_list: list of (num_other_nodes_b, 8) per sample in batch
        """
        B, E, C, N = edge_data.shape

        # --- Feature extraction ---
        x_flat = edge_data.view(B * E, C, N)                   # (B*E, 3, N)
        edge_emb = self.extractor(x_flat).view(B, E, self.d)   # (B, E, d)

        # --- Merge with edge type embeddings ---
        type_emb = self.edge_type_emb(edge_types)              # (B, E, d)
        edge_emb = self.edge_merge([edge_emb, type_emb])       # (B, E, d)

        # --- Self-attention (mask padding) ---
        # key_padding_mask: True = ignore (inverted from edge_mask)
        pad_mask = ~edge_mask  # (B, E)
        for attn in self.attn_layers:
            edge_emb = attn(edge_emb, key_padding_mask=pad_mask)

        # --- Edge classification head ---
        edge_logits = self.edge_head(edge_emb)  # (B, E, 2)

        # --- Node classification head ---
        node_logits_list = []
        for b in range(B):
            cols = cols_list[b]
            p = len(cols)
            col_idx = {name: i for i, name in enumerate(cols)}
            E_b = p * (p - 1)

            # Map (u, v) -> edge index in the flattened edge list
            edge_order = {}
            count = 0
            for ui in range(p):
                for vi in range(p):
                    if ui != vi:
                        edge_order[(ui, vi)] = count
                        count += 1

            x_idx = col_idx.get("X", None)
            y_idx = col_idx.get("Y", None)
            embs = edge_emb[b]  # (E, d)

            other_nodes = [
                name for name in cols if name not in ("X", "Y")
            ]
            if not other_nodes or x_idx is None or y_idx is None:
                node_logits_list.append(None)
                continue

            node_logits = []
            for node_name in other_nodes:
                u_idx = col_idx[node_name]
                # Gather 4 edges: u->X, u->Y, X->u, Y->u
                ux_emb = embs[edge_order[(u_idx, x_idx)]]
                uy_emb = embs[edge_order[(u_idx, y_idx)]]
                xu_emb = embs[edge_order[(x_idx, u_idx)]]
                yu_emb = embs[edge_order[(y_idx, u_idx)]]

                node_emb = self.node_merge(
                    [ux_emb, uy_emb, xu_emb, yu_emb]
                )  # (d,)
                logit = self.node_head(node_emb)  # (8,)
                node_logits.append(logit)

            if node_logits:
                node_logits_list.append(torch.stack(node_logits))  # (K, 8)
            else:
                node_logits_list.append(None)

        return edge_logits, node_logits_list

### Training (ModelWrapper)

In [9]:
# Cache at module level — only built once
# @crunch/keep:on
_GRAPH_LABEL, _ADJACENCY_LABEL = None, None

def get_adjacency_label():
    global _GRAPH_LABEL, _ADJACENCY_LABEL
    if _ADJACENCY_LABEL is None:
        _GRAPH_LABEL, _ADJACENCY_LABEL = create_graph_label()
    return _ADJACENCY_LABEL


def compute_class_weights(y_list: list) -> torch.Tensor:
    """Inverse-frequency weights for the 8 node classes."""
    adjacency_label = get_adjacency_label()
    counts = torch.zeros(N_CLASSES)
    for y_df in y_list:
        cols = list(y_df.columns)
        arr = y_df.values
        col_idx = {c: i for i, c in enumerate(cols)}
        xy = {"X", "Y"}
        for v in cols:
            if v in xy:
                continue
            vi, xi, yi = col_idx[v], col_idx["X"], col_idx["Y"]
            idx = [vi, xi, yi]
            sub = arr[np.ix_(idx, idx)]
            key = tuple(sub.flatten())
            label_str = adjacency_label.get(key, "Independent")
            counts[CLASS_NAMES.index(label_str)] += 1
    weights = 1.0 / (counts + 1e-6)
    return weights / weights.sum() * N_CLASSES


def compute_edge_class_weights(y_list: list) -> torch.Tensor:
    """Inverse-frequency weights for the 2 edge classes (no-edge vs edge)."""
    counts = torch.zeros(2)  # [no-edge, edge]
    for y_df in y_list:
        arr = y_df.values
        p = arr.shape[0]
        for i in range(p):
            for j in range(p):
                if i != j:
                    counts[int(arr[i, j])] += 1
    weights = 1.0 / (counts + 1e-6)
    return weights / weights.sum() * 2


class ADIAModelWrapper(pl.LightningModule):

    def __init__(
        self,
        d: int = D_MODEL,
        node_class_weights: torch.Tensor = None,
        edge_class_weights: torch.Tensor = None,
        lr: float = 1e-3,
        max_epochs: int = 20,
    ):
        super().__init__()
        self.model = ADIAModel(d)
        self.lr = lr
        self.max_epochs = max_epochs

        node_w = node_class_weights if node_class_weights is not None \
            else torch.ones(N_CLASSES)
        edge_w = edge_class_weights if edge_class_weights is not None \
            else torch.ones(2)

        self.node_criterion = nn.CrossEntropyLoss(weight=node_w)
        self.edge_criterion = nn.CrossEntropyLoss(weight=edge_w)

    def forward(self, batch):
        return self.model(
            batch["edge_data"].to(self.device),
            batch["edge_types"].to(self.device),
            batch["edge_mask"].to(self.device),
            batch["cols"],
        )

    def _compute_loss(self, batch, split: str):
        edge_logits, node_logits_list = self(batch)
        B = edge_logits.shape[0]
        edge_mask = batch["edge_mask"]
        cols_list = batch["cols"]
        adj_list = batch.get("adj", None)
        node_labels_list = batch.get("node_labels", None)

        total_loss = torch.tensor(0.0, device=self.device)
        n_terms = 0

        for b in range(B):
            cols = cols_list[b]
            p = len(cols)
            E_b = p * (p - 1)
            col_idx = {name: i for i, name in enumerate(cols)}

            # --- Edge classification loss ---
            if adj_list is not None:
                adj = adj_list[b].to(self.device)  # (p, p)
                edge_labels = []
                for ui in range(p):
                    for vi in range(p):
                        if ui != vi:
                            edge_labels.append(int(adj[ui, vi].item()))
                edge_labels = torch.tensor(
                    edge_labels, dtype=torch.long, device=self.device
                )
                e_logits = edge_logits[b, :E_b]  # (E_b, 2)
                loss_edge = self.edge_criterion(e_logits, edge_labels)
                total_loss = total_loss + loss_edge
                n_terms += 1

            # --- Node classification loss ---
            if node_labels_list is not None and node_logits_list[b] is not None:
                node_labels_dict = node_labels_list[b]
                other_nodes = [n for n in cols if n not in ("X", "Y")]
                node_labels_tensor = torch.tensor(
                    [node_labels_dict[n] for n in other_nodes],
                    dtype=torch.long, device=self.device,
                )
                n_logits = node_logits_list[b]  # (K, 8)
                loss_node = self.node_criterion(n_logits, node_labels_tensor)
                total_loss = total_loss + loss_node
                n_terms += 1

        if n_terms > 0:
            total_loss = total_loss / n_terms

        self.log(
            f"{split}_loss", total_loss,
            on_step=(split == "train"), on_epoch=True, prog_bar=True,
            batch_size=B,
        )
        return total_loss

    def training_step(self, batch, batch_idx):
        return self._compute_loss(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._compute_loss(batch, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.lr, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.max_epochs
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "epoch"}]

### Graph Utilities (same as baseline, kept intact)

In [10]:
def transform_proba_to_DAG(
    nodes: typing.List[str],
    pred: np.ndarray,
) -> np.ndarray:
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    G.add_edge("X", "Y")

    x_index, y_index = np.unravel_index(
        np.argsort(pred.ravel())[::-1], pred.shape
    )
    for i, j in zip(x_index, y_index):
        n1 = nodes[i]
        n2 = nodes[j]
        if i == j:
            continue
        if ((n1 == "X") and (n2 == "Y")) or ((n1 == "Y") and (n2 == "X")):
            continue
        if pred[i, j] > 0.5:
            G.add_edge(n1, n2)
            if not nx.is_directed_acyclic_graph(G):
                G.remove_edge(n1, n2)

    return nx.to_numpy_array(G)


def graph_nodes_representation(graph, nodelist):
    adjacency_matrix = nx.adjacency_matrix(
        graph, nodelist=nodelist
    ).todense()
    return tuple(adjacency_matrix.flatten())


def create_graph_label():
    graph_label = {
        nx.DiGraph([("X", "Y"), ("v", "X"), ("v", "Y")]): "Confounder",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("Y", "v")]): "Collider",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("v", "Y")]): "Mediator",
        nx.DiGraph([("X", "Y"), ("v", "X")]): "Cause of X",
        nx.DiGraph([("X", "Y"), ("v", "Y")]): "Cause of Y",
        nx.DiGraph([("X", "Y"), ("X", "v")]): "Consequence of X",
        nx.DiGraph([("X", "Y"), ("Y", "v")]): "Consequence of Y",
        nx.DiGraph({"X": ["Y"], "v": []}): "Independent",
    }
    nodelist = ["v", "X", "Y"]
    adjacency_label = {
        graph_nodes_representation(graph, nodelist): label
        for graph, label in graph_label.items()
    }
    return graph_label, adjacency_label


def get_labels(adjacency_matrix, adjacency_label):
    result = {}
    for variable in adjacency_matrix.columns.drop(["X", "Y"]):
        submatrix = adjacency_matrix.loc[
            [variable, "X", "Y"], [variable, "X", "Y"]
        ]
        key = tuple(submatrix.values.flatten())
        result[variable] = adjacency_label[key]
    return result

### Inference (node classification via the model)

In [11]:
# @crunch/keep:on
@torch.no_grad()
def infer_single(
    df: pd.DataFrame,
    model: ADIAModel,
    device: str = "cpu",
) -> pd.DataFrame:
    """
    Run inference on a single DataFrame.
    Returns adjacency matrix as pd.DataFrame using the node classification
    path for non-X/Y nodes and the edge classification path for X/Y edges.
    """
    model = model.eval()
    cols = list(df.columns)
    p = len(cols)
    nodes = cols

    edge_data, edge_types = build_edge_tensor(df)
    edge_data_t = torch.tensor(edge_data).unsqueeze(0).to(device)   # (1, E, 3, N)
    edge_types_t = torch.tensor(edge_types).unsqueeze(0).to(device) # (1, E)
    edge_mask_t = torch.ones(1, len(edge_types), dtype=torch.bool, device=device)

    edge_logits, node_logits_list = model(
        edge_data_t, edge_types_t, edge_mask_t, [cols]
    )

    # --- Build adjacency matrix from edge logits ---
    edge_probs = torch.softmax(edge_logits[0], dim=-1)[:, 1]  # (E,)
    E_mat = np.zeros((p, p))
    count = 0
    for i in range(p):
        for j in range(p):
            if i != j:
                E_mat[i, j] = edge_probs[count].item()
                count += 1

    adj = transform_proba_to_DAG(nodes, E_mat).astype(int)
    A = pd.DataFrame(adj, columns=nodes, index=nodes)

    # --- Override non-X/Y node classification with the dedicated head ---
    if node_logits_list[0] is not None:
        _, adjacency_label = create_graph_label()
        other_nodes = [n for n in cols if n not in ("X", "Y")]
        node_preds = torch.argmax(node_logits_list[0], dim=-1)  # (K,)
        col_idx = {name: i for i, name in enumerate(cols)}
        x_idx = col_idx.get("X")
        y_idx = col_idx.get("Y")

        for k, node_name in enumerate(other_nodes):
            pred_class = CLASS_NAMES[node_preds[k].item()]
            # Map class to subgraph pattern and update adjacency
            u_idx = col_idx[node_name]
            # Reset edges involving this node
            A.loc[node_name, :] = 0
            A.loc[:, node_name] = 0
            # Re-apply pattern
            patterns = {
                "Confounder":     [(node_name, "X"), (node_name, "Y")],
                "Collider":       [("X", node_name), ("Y", node_name)],
                "Mediator":       [("X", node_name), (node_name, "Y")],
                "Cause of X":     [(node_name, "X")],
                "Cause of Y":     [(node_name, "Y")],
                "Consequence of X": [("X", node_name)],
                "Consequence of Y": [("Y", node_name)],
                "Independent":    [],
            }
            for (src, dst) in patterns.get(pred_class, []):
                A.loc[src, dst] = 1

    return A

### CrunchDAO train / infer interface

In [12]:
# @crunch/keep:on
MAX_EPOCHS = 2

def train(
    X_train: typing.Dict[str, pd.DataFrame],
    y_train: typing.Dict[str, pd.DataFrame],
    model_directory_path: str,
) -> None:
    keys = list(X_train.keys())
    X_list = [X_train[k] for k in keys]
    y_list = [y_train[k] for k in keys]

    node_weights = compute_class_weights(y_list)
    edge_weights = compute_edge_class_weights(y_list)

    dataset = CausalEdgeDataset(X_list, y_list)

    wrapper = ADIAModelWrapper(
        d=D_MODEL,
        node_class_weights=node_weights,
        edge_class_weights=edge_weights,
        lr=1e-3,
        max_epochs=MAX_EPOCHS,
    )

    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        max_epochs=MAX_EPOCHS,
        precision="16-mixed",
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=True,
    )

    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True,
        drop_last=True,
        # num_workers=4,
        pin_memory=True,
        # persistent_workers=True,
        collate_fn=collate_fn,
    )
    trainer.fit(wrapper, loader)

    model_path = os.path.join(model_directory_path, "model.pt")
    torch.save(wrapper.model.state_dict(), model_path)
    print(f"Model saved to {model_path}")


def infer(
    X_test: typing.Dict[str, pd.DataFrame],
    model_directory_path: str,
    id_column_name: str,
    prediction_column_name: str,
) -> pd.DataFrame:
    model_path = os.path.join(model_directory_path, "model.pt")
    model = ADIAModel(d=D_MODEL)
    model.load_state_dict(
        torch.load(model_path, map_location="cpu", weights_only=True)
    )
    model.eval()
    device = "cpu"

    submission = {}
    for name in tqdm(X_test, desc="Inferring"):
        df = X_test[name]
        nodes = list(df.columns)
        A = infer_single(df, model, device)
        for i in nodes:
            for j in nodes:
                submission[f"{name}_{i}_{j}"] = int(A.loc[i, j])

    submission_series = pd.Series(submission)
    submission_df = submission_series.reset_index()
    submission_df.columns = [id_column_name, prediction_column_name]
    return submission_df

### Local Evaluation (mirrors baseline notebook)

### Local Training & Evaluation

In [13]:
import crunch

crunch = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>

cli version: 11.0.0
available ram: 31.00 gb
available cpu: 8 core
----


In [14]:
X_train, y_train, X_test = crunch.load_data()

data/y_test_reduced.pickle: download from https:crunchdao--competition--staging.s3-accelerate.amazonaws.com/data-releases/48/y_test_reduced.pickle (562930 bytes)
data/y_test_reduced.pickle: already exists, file length match
data/y_train.pickle: download from https:crunchdao--competition--staging.s3-accelerate.amazonaws.com/data-releases/48/y_train.pickle (7017735 bytes)
data/y_train.pickle: already exists, file length match
data/X_test_reduced.pickle: download from https:crunchdao--competition--staging.s3-accelerate.amazonaws.com/data-releases/48/X_test_reduced.pickle (122341879 bytes)
data/X_test_reduced.pickle: already exists, file length match
data/X_train.pickle: download from https:crunchdao--competition--staging.s3-accelerate.amazonaws.com/data-releases/48/X_train.pickle (1523944532 bytes)
data/X_train.pickle: already exists, file length match
data/example_prediction_reduced.parquet: download from https:crunchdao--competition--staging.s3-accelerate.amazonaws.com/data-releases/48/

In [15]:
def evaluate_local(
    X_train: dict,
    y_train: dict,
    test_size: float = 0.2,
    random_state: int = 42,
    max_samples: int = 200,
    max_epochs: int = 20,
):
    keys = list(X_train.keys())
    train_keys, test_keys = train_test_split(
        keys, test_size=test_size, random_state=random_state
    )
    print("building train/test split...")
    train_keys = train_keys[:max_samples * 10]
    test_keys = test_keys[:max_samples]
    print(f"Train keys: {len(train_keys)}, Test keys: {len(test_keys)}")

    eval_train_keys = train_keys[:max_samples * 10]
    eval_test_keys = test_keys[:max_samples]

    X_tr = [X_train[k] for k in eval_train_keys]
    y_tr = [y_train[k] for k in eval_train_keys]

    adjacency_label = get_adjacency_label()
    node_weights = compute_class_weights(y_tr)
    edge_weights = compute_edge_class_weights(y_tr)

    print("Building dataset and computing class weights...")
    dataset = CausalEdgeDataset(X_tr, y_tr)
    print(f"Training on {len(eval_train_keys)} samples, evaluating on {len(eval_test_keys)} samples...")

    wrapper = ADIAModelWrapper(
        d=D_MODEL,
        node_class_weights=node_weights,
        edge_class_weights=edge_weights,
        lr=1e-3,
        max_epochs=max_epochs,
    )

    trainer = pl.Trainer(
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        max_epochs=max_epochs,
        precision="16-mixed",
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=True,
    )

    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True,
        drop_last=True,
        # num_workers=4,
        pin_memory=True,
        # persistent_workers=True,
        collate_fn=collate_fn,
    )

    trainer.fit(wrapper, loader)

    # ---- Inference & scoring ----
    device = str(DEVICE)
    model = wrapper.model.to(device).eval()
    y_pred, y_true = [], []
    for name in tqdm(eval_test_keys, desc="Evaluating"):
        df = X_train[name]
        A = infer_single(df, model, device)
        predicted = get_labels(A, adjacency_label)
        ground_truth = get_labels(y_train[name], adjacency_label)
        for k in predicted:
            y_pred.append(predicted[k])
            y_true.append(ground_truth[k])

    y_pred = pd.Series(y_pred)
    y_true = pd.Series(y_true)
    scores = {}
    for label in y_true.unique():
        scores[label] = np.mean(y_pred[y_true == label] == label)
    scores = pd.Series(scores)
    scores["Balanced Accuracy"] = scores.mean()
    print(scores)
    return scores

In [16]:
# Quick local eval (reduce max_samples for speed)
scores = evaluate_local(X_train, y_train, max_samples=500, max_epochs=2)
print(scores)

building train/test split...
Train keys: 5000, Test keys: 500


Building dataset and computing class weights...
Building edge dataset (5000 samples, 7 workers)...


  0%|          | 0/5000 [00:00<?, ?it/s]

/drive1/phatnt/Graduation-Thesis-BsC-2026/.venv/lib/python3.12/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /drive1/phatnt/Graduation-Thesis-BsC-2026/.venv/lib ...
Using 16bit Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


...

/drive1/phatnt/Graduation-Thesis-BsC-2026/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/drive1/phatnt/Graduation-Thesis-BsC-2026/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]

Collider             0.193798
Independent          0.508997
Cause of Y           0.720670
Consequence of X     0.554479
Cause of X           0.241107
Mediator             0.021127
Consequence of Y     0.372222
Confounder           0.392157
Balanced Accuracy    0.375570
dtype: float64
Collider             0.193798
Independent          0.508997
Cause of Y           0.720670
Consequence of X     0.554479
Cause of X           0.241107
Mediator             0.021127
Consequence of Y     0.372222
Confounder           0.392157
Balanced Accuracy    0.375570
dtype: float64


### CrunchDAO Test & Submit

In [17]:
crunch.test(no_determinism_check=True)

print("Submit at: https://hub.crunchdao.com/competitions/causality-discovery/submit/via/notebook")

19:27:16 


19:27:16 started


19:27:16 running local test


19:27:16 internet access isn't restricted, no check will be done


19:27:16 


...

Building edge dataset (23500 samples, 7 workers)...


  0%|          | 0/23500 [00:00<?, ?it/s]

19:34:37 duration - time=00:07:20


19:34:37 memory - before="11.23 GB" after="25.68 GB" consumed="14.45 GB"


19:34:37 Cancelled!
